In [1]:
import pandas as pd
import os

In [2]:
path = '../processed/clean_data_monthly'
zone_lookup = pd.read_csv('../raw/taxi_zone_lookup.csv')

In [ ]:
frames = []
for file in os.listdir(path):
    if file.endswith(".parquet"):
        file_path = os.path.join(path, file)
        temp = pd.read_parquet(file_path, engine = "fastparquet")
        temp.drop(columns=[
            'store_and_fwd_flag',
            'RatecodeID',
            'payment_type',
        ], inplace=True)
        
        temp = temp.merge(
            zone_lookup[['LocationID','Zone']].rename(columns={'LocationID':'PULocationID','Zone':'PU_Zone'}),
            on='PULocationID', how='left'
        )
        
        temp = temp.merge(
            zone_lookup[['LocationID','Zone']].rename(columns={'LocationID':'DOLocationID','Zone':'DO_Zone'}),
            on='DOLocationID', how='left'
        )
        
        frames.append(temp)
df = pd.concat(frames)

## Feature engine

In [4]:
df["pickup_date"] = df["tpep_pickup_datetime"].dt.date
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_dow"] = df["tpep_pickup_datetime"].dt.day_name().astype('category')
df["pickup_month"] = df["tpep_pickup_datetime"].dt.month

## Tính KPI theo Địa điểm đón

In [5]:
kpi_PU = df.groupby('PU_Zone').agg(
        total_trips=('tpep_pickup_datetime', 'size'), # Tính tổng số chuyến đi
        total_revenue=('total_amount', 'sum'),  # Tính tổng doanh thu
        total_distance_miles=('trip_distance', 'sum'),  # Tính tổng khoảng cách chuyến đi
        median_duration_minutes=('trip_duration_minutes', 'median'),  #Tính p50 thời gian đi
        p95_duration_minutes=('trip_duration_minutes', lambda x: x.quantile(0.95)),
        median_speed_mph=('speed_mph', 'median'), # p50 speed
        avg_passengers=('passenger_count', 'mean') # Trung bình số người đi
    )
kpi_PU['trips_per_day'] = kpi_PU['total_trips'] / 365 # Trung bình số chuyến trên ngày

In [6]:
PU_path = f'processed/kpi_PU_2023.csv'
# kpi_PU.to_csv(PU_path, encoding='utf-8-sig')
print(f"Đã lưu KPI theo PU: {PU_path}")

Đã lưu KPI theo PU: processed/kpi_PU_2023.csv


## Tính KPI theo Địa điểm đến

In [7]:
kpi_DO = df.groupby('DO_Zone').agg(
        total_trips=('tpep_pickup_datetime', 'size'), # Tính tổng số chuyến đi
        total_revenue=('total_amount', 'sum'),  # Tính tổng doanh thu
        total_distance_miles=('trip_distance', 'sum'),  # Tính tổng khoảng cách chuyến đi
        median_duration_minutes=('trip_duration_minutes', 'median'),  #Tính p50 thời gian đi
        p95_duration_minutes=('trip_duration_minutes', lambda x: x.quantile(0.95)),
        median_speed_mph=('speed_mph', 'median'), # p50 speed
        avg_passengers=('passenger_count', 'mean') # Trung bình số người đi
    )
kpi_DO['trips_per_day'] = kpi_DO['total_trips'] / 365 # Trung bình số chuyến trên ngày

In [8]:
DO_path = f'processed/kpi_DO_2023.csv'
# kpi_DO.to_csv(DO_path, encoding='utf-8-sig')
print(f"Đã lưu KPI theo PU: {DO_path}")

Đã lưu KPI theo PU: processed/kpi_DO_2023.csv


## Tính KPI theo ngày

In [9]:
kpi_daily = df.groupby('pickup_date').agg(
        total_trips=('tpep_pickup_datetime', 'size'), # Tính tổng số chuyến đi
        total_revenue=('total_amount', 'sum'),  # Tính tổng doanh thu
        total_distance_miles=('trip_distance', 'sum'),  # Tính tổng khoảng cách chuyến đi
        median_duration_minutes=('trip_duration_minutes', 'median'),  #Tính p50 thời gian đi
        p95_duration_minutes=('trip_duration_minutes', lambda x: x.quantile(0.95)),
        median_speed_mph=('speed_mph', 'median'), # p50 speed
        avg_passengers=('passenger_count', 'mean'), # Trung bình số người đi
        most_pickedup_place=('PU_Zone', lambda x: x.value_counts().idxmax()),
        most_visited_place=('DO_Zone', lambda x: x.value_counts().idxmax())
    )

avg_daily_trips = kpi_daily['total_trips'].mean() # Tính trung bình số chuyến đi trong 1 ngày
kpi_daily['index_trips_100'] = round((kpi_daily['total_trips'] / avg_daily_trips) * 100, 2) # công thức này sẽ cho biết tỉ lệ so với chuyến đi trung bình trong ngày (Index(100) theo ngày trong yêu cầu)

Lưu file

In [10]:
daily_path = f'processed/kpi_daily_2023.csv'
# kpi_daily.to_csv(daily_path, encoding='utf-8-sig')
print(f"Đã lưu KPI theo ngày: {daily_path}")

Đã lưu KPI theo ngày: processed/kpi_daily_2023.csv


## TÍnh KPI theo tuần

In [11]:
dow_day_counts = df.groupby('pickup_dow', observed=True)['pickup_date'].nunique() # Lọc ra những ngày khác biệt theo thứ

kpi_dow = df.groupby('pickup_dow', observed=True).agg(
        total_trips=('tpep_pickup_datetime', 'size'), # sum
        total_revenue=('total_amount', 'sum'), # sum
        median_duration_minutes=('trip_duration_minutes', 'median'), # p50 duration
        p95_duration_minutes=('trip_duration_minutes', lambda x: x.quantile(0.95)),  # p95(Phân vị 95%): 95% dữ liệu nhỏ hơn giá trị này
        median_speed_mph=('speed_mph', 'median')   # p50: Phân vị 50%, tức là median
)

# Chuẩn hóa để có số liệu trung bình/ngày vì dùng total_trip thì chủ nhật có 53 cái thì sẽ không phán ánh đúng
kpi_dow['avg_trips_per_day'] = kpi_dow['total_trips'] / dow_day_counts # Tổng số trip / số lượng ngày của thứ đó
kpi_dow['avg_revenue_per_day'] = kpi_dow['total_revenue'] / dow_day_counts

# Tương tự như kpi theo ngày (Indexdow theo thứ trong tuần)
avg_dow_trips_overall = kpi_dow['avg_trips_per_day'].mean()
kpi_dow['index_dow_trips'] = round((kpi_dow['avg_trips_per_day'] / avg_dow_trips_overall) * 100, 2)

# Sắp xếp lại thứ tự thứ 
days_of_week_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
kpi_dow = kpi_dow.reindex(days_of_week_order)

Lưu file

In [12]:
weekly_path = f'processed/kpi_weekly_2023.csv'
# kpi_dow.to_csv(weekly_path, encoding='utf-8-sig')
print(f"Đã lưu KPI theo thứ trong tuần: {weekly_path}")

Đã lưu KPI theo thứ trong tuần: processed/kpi_weekly_2023.csv


## Tính KPI theo tháng

In [13]:
kpi_monthly = df.groupby('pickup_month').agg(
    total_trips=('tpep_pickup_datetime', 'size'), # sum
    total_revenue=('total_amount', 'sum'), # sum
    total_miles=('trip_distance', 'sum'), # Tổng quãng đường
    median_duration_minutes=('trip_duration_minutes', 'median'), # p50 duration
    p95_duration_minutes=('trip_duration_minutes', lambda x: x.quantile(0.95)),  # p95
    median_speed_mph=('speed_mph', 'median')
)   

In [14]:
# Tính % tổng năm theo tháng (% số chuyến đi, % doanh thu revenue)
total_annual_trips = kpi_monthly['total_trips'].sum()
kpi_monthly['percent_of_annual_trips'] = round((kpi_monthly['total_trips'] / total_annual_trips) * 100, 2)

total_annual_revenue = kpi_monthly['total_revenue'].sum()
kpi_monthly['percent_of_annual_revenue'] = round((kpi_monthly['total_revenue'] / total_annual_revenue) * 100, 2)

Lưu file

In [15]:
monthly_path = f'processed/kpi_monthly_2023.csv'
# kpi_monthly.to_csv(monthly_path, encoding='utf-8-sig')
print(f"Đã lưu KPI theo thứ trong tuần: {monthly_path}")

Đã lưu KPI theo thứ trong tuần: processed/kpi_monthly_2023.csv


In [16]:
# Đếm số ngày duy nhất trong bộ dữ liệu để tính trung bình
total_days_in_dataset = df['pickup_date'].nunique()

kpi_hourly = df.groupby('pickup_hour').agg(
    total_trips=('tpep_pickup_datetime', 'size'),
    median_speed_mph=('speed_mph', 'median'),
    median_duration_min=('trip_duration_minutes', 'median'),
    p95_distance_miles=('trip_distance', lambda x: x.quantile(0.95))
)

Lưu file

In [17]:
hourly_path = f'processed/kpi_hourly_2023.csv'
# kpi_hourly.to_csv(hourly_path, encoding='utf-8-sig')
print(f"Đã lưu KPI theo giờ trong ngày: {hourly_path}")

Đã lưu KPI theo giờ trong ngày: processed/kpi_hourly_2023.csv
